# 第三章：LCEL 链式表达式

## 学习目标
- 理解 LCEL（LangChain Expression Language）解决什么问题
- 掌握 `|` 管道操作符串联组件
- 使用 Output Parser 结构化输出
- 学会 `RunnablePassthrough`、`RunnableLambda`、`RunnableParallel`
- 实现容错回退（Fallback）

## 0. 环境准备

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")
import langchain
print(f"langchain 版本: {langchain.__version__}")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="kimi-k2.5",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
    temperature=0.7,
)
print("模型初始化完成")

## 1. 对比：手动串联 vs LCEL

在第二章中，我们一直在手动调用 `prompt.invoke()` 再传给 `llm.invoke()`。先看这种写法有什么不便：

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一位{domain}专家，用一句话回答问题。"),
    ("human", "{question}"),
])

# ❌ 手动串联：每一步都要手动传递
messages = prompt.invoke({"domain": "天文学", "question": "为什么天空是蓝色的？"})
ai_message = llm.invoke(messages)
text = ai_message.content

print(text)



因为大气分子对短波长蓝光的散射能力比长波长红光强得多，阳光经瑞利散射后布满天空。


In [6]:
# ✅ LCEL：用 | 管道一行搞定
chain = prompt | llm | StrOutputParser()

result = chain.invoke({"domain": "天文学", "question": "为什么天空是蓝色的？"})
print(result)



天空呈现蓝色主要是由于**瑞利散射**效应：太阳光进入大气层后，其波长较短的蓝光（以及部分紫光）被大气分子（如氮气、氧气）向四面八方散射，因此我们看到的天空充满了这些被散射的光线，从而呈现蓝色。


### 刚才发生了什么？

`prompt | llm | StrOutputParser()` 创建了一条 **链（chain）**，数据从左到右流动：

```
输入字典 → prompt（生成消息列表）→ llm（生成 AIMessage）→ StrOutputParser（提取纯文本）→ 字符串
```

等价于 `StrOutputParser().invoke(llm.invoke(prompt.invoke(input)))`，但可读性好得多。

更重要的是，`chain` 本身也是一个 Runnable，自动获得 `.invoke()` / `.stream()` / `.batch()` 方法——不需要额外代码。

### LCEL 的核心优势

| 优势 | 说明 |
|------|------|
| 可读性 | 数据流向一目了然：从左到右 |
| 统一接口 | 链自动获得 invoke / stream / batch |
| 流式透传 | 中间步骤支持流式，首 token 延迟更低 |
| 可组合 | 链可以嵌套到更大的链中 |

In [7]:
# 链也支持流式——不需要任何额外代码
for chunk in chain.stream({"domain": "生物学", "question": "DNA 和 RNA 有什么区别？"}):
    print(chunk, end="", flush=True)

print()



DNA是双链、脱氧核糖和胸腺嘧啶的分子，主要负责遗传信息的储存与复制；而RNA是单链、核糖和尿嘧啶的分子，主要在转录和翻译中传递遗传信息并参与蛋白质合成。


In [8]:
# 链也支持批量
results = chain.batch([
    {"domain": "物理学", "question": "什么是量子纠缠？"},
    {"domain": "化学", "question": "什么是共价键？"},
])

for r in results:
    print(r)
    print("-" * 40)



量子纠缠是指两个或多个粒子在量子态上相互关联，使得对其中一个粒子的测量会瞬间决定另一个粒子的状态，无论它们相隔多远。
----------------------------------------


共价键是两个原子通过共享电子对形成的化学键，使原子获得稳定的外层电子构型。
----------------------------------------


## 2. Output Parser（输出解析器）

上面用了 `StrOutputParser` 把 AIMessage 转成字符串。LangChain 还提供了其他解析器：

In [9]:
from langchain_core.output_parsers import JsonOutputParser

json_prompt = ChatPromptTemplate.from_messages([
    ("system", """从用户输入中提取结构化信息，以 JSON 格式输出。
字段：name（姓名）、age（年龄）、skills（技能列表）。
只输出 JSON，不要其他内容。"""),
    ("human", "{text}"),
])

json_chain = json_prompt | llm | JsonOutputParser()

result = json_chain.invoke({"text": "小明今年 25 岁，会 Python、Java 和机器学习"})
print(type(result))  # dict
print(result)

<class 'dict'>
{'name': '小明', 'age': 25, 'skills': ['Python', 'Java', '机器学习']}


### 常用 Output Parser

| 解析器 | 输入 | 输出 | 用途 |
|--------|------|------|------|
| `StrOutputParser` | AIMessage | `str` | 提取纯文本 |
| `JsonOutputParser` | AIMessage | `dict` | 解析 JSON |
| `PydanticOutputParser` | AIMessage | Pydantic 对象 | 带类型校验的结构化输出 |
| `CommaSeparatedListOutputParser` | AIMessage | `list[str]` | 逗号分隔列表 |

### 常见问题

- **`JsonOutputParser` 报错**：模型返回的不是合法 JSON（可能带了 markdown 代码块标记）。可以在 system prompt 中明确要求「只输出 JSON，不要 markdown 代码块」。
- **解析失败不会重试**：默认情况下解析失败直接抛异常。需要重试可以用 `.with_retry()`，或用 fallback（后面会讲）。

## 3. RunnablePassthrough（透传与追加字段）

`RunnablePassthrough` 将输入原样传递到下一步。单独用没什么意义，配合 `.assign()` 可以在不改变原始输入的情况下追加新字段。

In [10]:
from langchain_core.runnables import RunnablePassthrough

# assign() 在输入字典基础上追加字段
chain = RunnablePassthrough.assign(
    char_count=lambda x: len(x["text"])
)

result = chain.invoke({"text": "Hello, LangChain!"})
print(result)
# {'text': 'Hello, LangChain!', 'char_count': 17}

{'text': 'Hello, LangChain!', 'char_count': 17}


In [11]:
# 实际用途：在链中动态计算上下文信息
analysis_prompt = ChatPromptTemplate.from_messages([
    ("system", "分析以下文本（共 {char_count} 个字符）的情感倾向，只回答'正面'、'负面'或'中性'。"),
    ("human", "{text}"),
])

analysis_chain = (
    RunnablePassthrough.assign(char_count=lambda x: len(x["text"]))
    | analysis_prompt
    | llm
    | StrOutputParser()
)

result = analysis_chain.invoke({"text": "这个产品质量非常好，用起来很舒服！"})
print(result)



正面


### 刚才发生了什么？

`RunnablePassthrough.assign(char_count=...)` 接收 `{"text": "..."}` 输入，保留原字段并追加 `char_count`，变成 `{"text": "...", "char_count": 17}`。然后交给模板填充变量。

这在 RAG 场景中特别有用——检索到的文档需要作为新字段追加到输入中。

## 4. RunnableLambda（自定义函数）

将任意 Python 函数包装为 Runnable，插入链中。

In [12]:
from langchain_core.runnables import RunnableLambda

# 自定义预处理
def preprocess(input_dict):
    """清理用户输入"""
    return {
        "text": input_dict["text"].strip(),
        "language": input_dict.get("language", "中文"),
    }

# 自定义后处理
def postprocess(output_str):
    return f"翻译结果：{output_str}"

translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "将用户输入翻译成{language}，只输出翻译结果。"),
    ("human", "{text}"),
])

# 完整链：预处理 → 模板 → 模型 → 提取文本 → 后处理
translate_chain = (
    RunnableLambda(preprocess)
    | translate_prompt
    | llm
    | StrOutputParser()
    | RunnableLambda(postprocess)
)

result = translate_chain.invoke({"text": "  人工智能正在快速发展  ", "language": "英语"})
print(result)

翻译结果：

Artificial intelligence is developing rapidly


### 刚才发生了什么？

数据流经 5 个步骤，每一步都是一个 Runnable：

```
原始输入 → preprocess（清理空格） → prompt（生成消息）→ llm（生成回复）→ StrOutputParser（提取文本）→ postprocess（加前缀）
```

`RunnableLambda` 让你可以在链中的任意位置插入自定义逻辑，不受 LangChain 内置组件的限制。

## 5. RunnableParallel（并行执行）

同时运行多条链，将结果合并为一个字典：

```
输入 ──┬── chain_a ──> result_a
       ├── chain_b ──> result_b  → 合并 → {"a": ..., "b": ..., "c": ...}
       └── chain_c ──> result_c
```

In [13]:
from langchain_core.runnables import RunnableParallel

# 三条独立的分析链，并行执行
parallel_chain = RunnableParallel(
    sentiment=ChatPromptTemplate.from_template(
        "分析情感倾向，只回答'正面'、'负面'或'中性'：\n{text}"
    ) | llm | StrOutputParser(),

    summary=ChatPromptTemplate.from_template(
        "用一句话概括核心内容：\n{text}"
    ) | llm | StrOutputParser(),

    keywords=ChatPromptTemplate.from_template(
        "提取 3 个关键词，用逗号分隔：\n{text}"
    ) | llm | StrOutputParser(),
)

result = parallel_chain.invoke({
    "text": "LangChain 极大地简化了大语言模型应用的开发流程，让开发者可以专注于业务逻辑。"
})

for key, value in result.items():
    print(f"{key}: {value}")

sentiment: 

正面
summary: 

LangChain 通过封装底层复杂性，让开发者能更专注于业务逻辑，从而大幅简化了大语言模型应用的开发。
keywords: 

LangChain, 大语言模型, 开发流程


### 字典语法简写

在 LCEL 中，直接传字典等价于 `RunnableParallel`：

In [14]:
# 字典语法 = RunnableParallel 的简写
chain = {
    "en": ChatPromptTemplate.from_template("翻译成英语，只输出结果：{text}")
          | llm | StrOutputParser(),
    "ja": ChatPromptTemplate.from_template("翻译成日语，只输出结果：{text}")
          | llm | StrOutputParser(),
}

result = RunnableParallel(chain).invoke({"text": "知识就是力量"})
print("英语:", result["en"])
print("日语:", result["ja"])

英语: 

Knowledge is power
日语: 

知識は力である


三条链共享同一个输入，**并行**发送 API 请求，总耗时约等于最慢的那条，而非三条之和。

## 6. 链的嵌套（多步处理）

链可以作为更大链的一部分。下面演示一个两步工作流：先提取关键词，再基于关键词生成大纲。

In [15]:
# 第一步：提取关键词
step1 = (
    ChatPromptTemplate.from_template(
        "从以下主题中提取 3 个核心关键词，用逗号分隔，只输出关键词：\n{topic}"
    )
    | llm
    | StrOutputParser()
)

# 第二步：基于关键词生成大纲
step2_prompt = ChatPromptTemplate.from_template(
    """基于以下关键词，为一篇技术博客生成大纲（标题 + 3-5 个章节）。

关键词：{keywords}
原始主题：{topic}"""
)

# 组合：step1 输出赋给 keywords，原始 topic 透传
full_chain = (
    RunnablePassthrough.assign(keywords=step1)
    | step2_prompt
    | llm
    | StrOutputParser()
)

result = full_chain.invoke({"topic": "如何用 RAG 技术构建企业知识库"})
print(result)



# 如何用 RAG 技术构建企业知识库

## 章节大纲

**一、引言：企业知识管理的痛点与 RAG 的机遇**
- 传统知识库搜索的局限性
- RAG 技术如何重塑企业知识访问方式
- 构建智能知识库的预期价值

**二、RAG 技术核心原理**
- 检索增强生成的基本架构
- 向量检索与语义匹配机制
- 大语言模型与知识库的协同工作流程

**三、企业知识库构建全流程**
- 知识采集与数据预处理策略
- 文档分块与向量化方法
- 向量数据库选型与部署
- 知识索引的优化与维护

**四、RAG 系统关键技术实现**
- 检索模块设计：混合检索策略
- 生成模块优化：提示词工程与上下文管理
- 检索与生成的协同调优
- 企业级安全与权限控制

**五、落地实践：最佳实践与常见挑战**
- 不同行业知识库的差异化构建
- 性能优化与成本控制
- 持续迭代与效果评估方法
- 未来演进方向展望


### 刚才发生了什么？

这是本章最关键的组合技巧：

1. `RunnablePassthrough.assign(keywords=step1)` 做了两件事：
   - 把原始输入 `{"topic": "..."}` 原样透传
   - 同时把输入传给 `step1` 链，将其输出作为 `keywords` 追加
   - 结果：`{"topic": "...", "keywords": "RAG, 向量数据库, ..."}`

2. 这个字典传给 `step2_prompt`，两个变量都被填充。

这就是 LCEL 的核心模式：**用 `assign` 连接多步之间的数据依赖**。

## 7. Fallback（容错回退）

`.with_fallbacks()` 在主链失败时自动切换到备用链。

In [ ]:
# 模拟一个会失败的模型（错误的 API key）
bad_llm = ChatOpenAI(
    model="kimi-k2.5",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key="sk-invalid-key",
)

# 主模型失败时，自动切换到备用模型
llm_with_fallback = bad_llm.with_fallbacks([llm])

response = llm_with_fallback.invoke("你好")
print(response.content)  # bad_llm 失败，自动用 llm 返回结果

In [17]:
# 在链中使用 fallback
robust_chain = (
    ChatPromptTemplate.from_template("用一句话解释：{concept}")
    | llm_with_fallback
    | StrOutputParser()
)

result = robust_chain.invoke({"concept": "微服务架构"})
print(result)



微服务架构是一种将大型应用程序拆分为多个独立、可部署的小型服务的设计方式，每个服务专注于单一业务功能，并通过轻量级通信协议（如HTTP/REST或消息队列）相互协作，以实现整体系统的灵活性和可维护性。


### 典型应用场景

- **模型降级**：主力模型超时或限流时，自动切换到更快的模型（如 qwen-max → qwen-turbo）
- **解析容错**：JSON 解析失败时，退回到纯文本输出

### 常见问题

- **所有 fallback 都失败**：会抛出最后一个 fallback 的异常。确保至少有一个可靠的兜底选项。
- **性能开销**：fallback 是串行尝试的，失败后才会尝试下一个，会增加延迟。

## 8. 查看链的结构

In [18]:
!uv pip install grandalf

Using Python 3.11.15 environment at: /home/sbh/Agent/LangChain/.venv
Checked 1 package in 2ms


In [21]:
# 用 .get_graph().print_ascii() 可视化链结构
simple_chain = (
    ChatPromptTemplate.from_template("解释：{concept}")
    | llm
    | StrOutputParser()
)

simple_chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


In [22]:
# 并行链的结构
parallel_chain.get_graph().print_ascii()

                        +-------------------------------------------+                          
                        | Parallel<sentiment,summary,keywords>Input |                          
                        +-------------------------------------------+                          
                          *******              *              *******                          
                     *****                     *                     *****                     
                 ****                          *                          ****                 
+--------------------+              +--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+              +--------------------+ 
           *                                   *                                   *           
           *                            

调试时用 `print_ascii()` 可以快速确认链的拓扑是否符合预期。

## 9. 实战：文本分析流水线

综合运用本章知识，构建一条完整的分析链。

In [23]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

# 并行分析：情感 + 摘要 + 翻译 同时进行
parallel_analysis = RunnableParallel(
    summary=ChatPromptTemplate.from_template("用一句话概括：\n{text}")
            | llm | StrOutputParser(),
    sentiment=ChatPromptTemplate.from_template("判断情感（正面/负面/中性），只输出结果：\n{text}")
            | llm | StrOutputParser(),
    translation=ChatPromptTemplate.from_template("翻译成英语，只输出译文：\n{text}")
            | llm | StrOutputParser(),
)

# 后处理：汇总为报告
def format_report(result):
    return (
        f"文本分析报告\n"
        f"{'='*40}\n"
        f"摘要：{result['summary']}\n"
        f"情感：{result['sentiment']}\n"
        f"英译：{result['translation']}\n"
        f"{'='*40}"
    )

pipeline = parallel_analysis | RunnableLambda(format_report)

report = pipeline.invoke({
    "text": "LangChain 极大地简化了大语言模型应用的开发流程，让开发者可以专注于业务逻辑而非底层细节。"
})

print(report)

文本分析报告
摘要：

LangChain 简化了大语言模型应用的开发，让开发者能专注于业务逻辑而无需顾虑底层实现细节。
情感：

正面
英译：

LangChain greatly simplifies the development process of large language model applications, allowing developers to focus on business logic rather than underlying details.


这条链的数据流：

```
输入 → RunnableParallel ──┬── summary 链
                          ├── sentiment 链   → 合并字典 → format_report → 最终报告
                          └── translation 链
```

三条分析链并行执行，结果汇总后经过自定义函数格式化。整个流水线用一次 `.invoke()` 调用完成。

## 总结

本章学习了：
- ✅ 对比了手动串联 vs LCEL `|` 管道语法
- ✅ `StrOutputParser` 和 `JsonOutputParser` 结构化输出
- ✅ `RunnablePassthrough.assign()` 透传输入并追加字段
- ✅ `RunnableLambda` 将自定义函数插入链
- ✅ `RunnableParallel` 并行执行多条链
- ✅ `.with_fallbacks()` 容错回退
- ✅ `.get_graph().print_ascii()` 可视化链结构

### 速查表

| 组件 | 作用 | 典型用法 |
|------|------|----------|
| `\|` | 串联 | `prompt \| llm \| parser` |
| `StrOutputParser` | AIMessage → str | 提取纯文本 |
| `JsonOutputParser` | AIMessage → dict | 结构化输出 |
| `RunnablePassthrough` | 透传 + 追加字段 | `.assign(key=chain)` 连接多步依赖 |
| `RunnableLambda` | 包装函数 | 预处理 / 后处理 |
| `RunnableParallel` | 并行 | 多维度分析，字典语法简写 |
| `.with_fallbacks()` | 容错 | 模型降级 |

## 下一章

**第四章：Memory（对话历史与状态管理）** —— 第二章中我们手动构造 `history` 消息列表传给 `MessagesPlaceholder`，下一章将学习如何让 LangChain 自动管理对话历史，实现真正的多轮对话。